# Notebook 03 — Camera Models, Calibration & Projection

**Vision & 3D Mapping Workshop** | Block 1: Mathematical Foundations

---

## Why This Matters

Every 3D mapping system — from autonomous drones to AR headsets — must answer two
fundamental questions:

1. **Projection**: Given a 3D point in the world, where does it appear in the image?
2. **Back-projection**: Given a pixel and its depth, what 3D point does it correspond to?

The **pinhole camera model** provides the mathematical framework for both. It is the
foundation upon which Structure from Motion (Notebook 07), stereo depth (Notebook 06),
visual odometry (Notebook 10), and neural 3D reconstruction (Notebooks 11–12) are all built.

**Key insight**: The projection equation $\mathbf{p} = K[R|\mathbf{t}]\mathbf{P}$ is a
**lossy** operation — it collapses 3D to 2D, destroying depth. The entire field of 3D
computer vision is fundamentally about *recovering* that lost depth.

### What You'll Learn

1. **The Pinhole Camera Model** — projection from first principles
2. **Back-Projection** — the inverse: recovering 3D from 2D + depth
3. **Lens Distortion** — radial and tangential models
4. **Camera Calibration** — estimating intrinsics via Zhang's method
5. **The Camera Matrix** — decomposition and degrees of freedom
6. **Field of View** — the viewing frustum
7. **Fisheye Models** — Kannala-Brandt and Double Sphere for wide-angle lenses

### Prerequisites
- Linear algebra (Notebook 02: homogeneous coordinates, projective geometry)
- NumPy familiarity

### References
- Hartley & Zisserman, "Multiple View Geometry", Ch. 6
- Szeliski, "Computer Vision: Algorithms and Applications" 2nd ed, Ch. 2
- Zhang, "A Flexible New Technique for Camera Calibration", TPAMI 2000
- Kannala & Brandt, "A Generic Camera Model and Calibration Method for Conventional, Wide-Angle, and Fish-Eye Lenses", TPAMI 2006
- Usenko et al., "The Double Sphere Camera Model", 3DV 2018

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import cv2

np.set_printoptions(precision=4, suppress=True)
%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100

---

## Section 1: The Pinhole Camera Model

### 1.1 From Light Rays to Geometry

The simplest camera model is the **pinhole camera**: all light rays pass through a single
point (the **optical center** or **center of projection**) before hitting the image plane.

Consider a 3D point $\mathbf{P} = (X, Y, Z)^\top$ in camera coordinates, where $Z$ is the
depth (distance along the optical axis). By **similar triangles**, the projected point on the
image plane at distance $f$ (the focal length) from the pinhole is:

\[
x = f \frac{X}{Z}, \qquad y = f \frac{Y}{Z}
\]

This is the fundamental **perspective projection** equation.

### 1.2 The Intrinsic Matrix $K$

To convert from metric coordinates $(x, y)$ on the image plane to **pixel coordinates**
$(u, v)$, we need to account for:

- **Focal length in pixels**: $f_x = f \cdot m_x$, $f_y = f \cdot m_y$ where $m_x, m_y$
  are pixels per millimeter along each axis
- **Principal point**: $(c_x, c_y)$ — the pixel where the optical axis intersects the
  image plane (ideally the image center)

The mapping from camera-frame 3D to pixel coordinates is:

\[
u = f_x \frac{X}{Z} + c_x, \qquad v = f_y \frac{Y}{Z} + c_y
\]

In **homogeneous coordinates**, this becomes a clean matrix multiplication:

\[
\lambda \begin{pmatrix} u \\ v \\ 1 \end{pmatrix}
= \underbrace{\begin{pmatrix} f_x & 0 & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{pmatrix}}_{K}
\begin{pmatrix} X \\ Y \\ Z \end{pmatrix}
\]

where $\lambda = Z$ is the depth and $K$ is the **camera intrinsic matrix** (also called the
**calibration matrix**). It encodes 5 intrinsic parameters: $f_x, f_y, c_x, c_y$, and
optionally skew $s$ (zero for modern cameras).

### 1.3 The Full Projection Equation

To project a point $\mathbf{P}_w = (X_w, Y_w, Z_w, 1)^\top$ from **world coordinates**
to pixel coordinates, we first transform to camera coordinates via the **extrinsic parameters**
$(R, \mathbf{t})$, then apply the intrinsic matrix:

\[
\lambda \begin{pmatrix} u \\ v \\ 1 \end{pmatrix}
= K \begin{pmatrix} R & \mathbf{t} \end{pmatrix}
\begin{pmatrix} X_w \\ Y_w \\ Z_w \\ 1 \end{pmatrix}
= K [R | \mathbf{t}] \mathbf{P}_w
\]

where:
- $R \in SO(3)$ is the $3 \times 3$ rotation matrix (camera orientation)
- $\mathbf{t} \in \mathbb{R}^3$ is the translation vector (camera position offset)
- $[R | \mathbf{t}]$ is the $3 \times 4$ **extrinsic matrix**

The combined $3 \times 4$ matrix $P = K[R|\mathbf{t}]$ is the **camera projection matrix**.

> **Note:** The **3×4** form $[\mathbf{R} \mid \mathbf{t}]$ is the projection matrix extrinsics (used for $\mathbf{x} = K[R \mid t]\mathbf{X}$). The **4×4** homogeneous form $T = \begin{bsmallmatrix} R & t \\ 0 & 1 \end{bsmallmatrix} \in SE(3)$ (introduced in NB04) is used for composing transforms and Lie group operations.

### 1.4 Physical Meaning of Parameters

| Parameter | Meaning | Typical Value |
|-----------|---------|---------------|
| $f_x, f_y$ | Focal length in pixels = $f_{\text{mm}} \times \text{px/mm}$ | 500–2000 px |
| $c_x, c_y$ | Principal point (optical axis pierces sensor) | $\approx W/2, H/2$ |
| $R$ | Rotation from world to camera frame | $\in SO(3)$ |
| $\mathbf{t}$ | Translation: $\mathbf{t} = -R\mathbf{C}$ where $\mathbf{C}$ is camera center | $\in \mathbb{R}^3$ |

> **Similar-triangles derivation:** Consider a 3D point $P = (X, Y, Z)$ and a pinhole camera with focal length $f$ and the image plane at distance $f$ from the pinhole center. By similar triangles in the $XZ$-plane:
>
> $$\frac{x_{\text{image}}}{f} = \frac{X}{Z} \quad \Longrightarrow \quad x_{\text{image}} = f \cdot \frac{X}{Z}$$
>
> Similarly in the $YZ$-plane: $y_{\text{image}} = f \cdot Y/Z$. Adding the principal point offset $(c_x, c_y)$ and allowing different focal lengths $(f_x, f_y)$ per axis:
>
> $$u = f_x \frac{X}{Z} + c_x, \quad v = f_y \frac{Y}{Z} + c_y$$
>
> In matrix form: $\lambda \begin{pmatrix}u\\v\\1\end{pmatrix} = \begin{pmatrix}f_x & 0 & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1\end{pmatrix} \begin{pmatrix}X\\Y\\Z\end{pmatrix}$ where $\lambda = Z$ is the projective depth.

In [ ]:
def make_intrinsic(fx, fy, cx, cy):
    """Construct the 3x3 camera intrinsic matrix K."""
    return np.array([
        [fx,  0, cx],
        [ 0, fy, cy],
        [ 0,  0,  1]
    ], dtype=np.float64)


def project_points(P_world, K, R, t):
    """
    Project 3D world points onto the image plane.

    Parameters
    ----------
    P_world : (N, 3) array — 3D points in world coordinates
    K       : (3, 3) array — intrinsic matrix
    R       : (3, 3) array — rotation (world → camera)
    t       : (3,)  array — translation

    Returns
    -------
    pixels : (N, 2) array — projected pixel coordinates (u, v)
    depths : (N,)   array — depth of each point in camera frame
    """
    P_cam = (R @ P_world.T + t[:, None]).T  # (N, 3) in camera frame
    depths = P_cam[:, 2].copy()

    p_hom = (K @ P_cam.T).T                 # (N, 3) homogeneous pixel coords
    pixels = p_hom[:, :2] / p_hom[:, 2:3]   # dehomogenize

    return pixels, depths

In [ ]:
def make_cube(center=(0, 0, 5), size=1.0):
    """Generate the 8 vertices and 12 edges of a 3D cube."""
    s = size / 2
    cx, cy, cz = center
    verts = np.array([
        [cx-s, cy-s, cz-s], [cx+s, cy-s, cz-s],
        [cx+s, cy+s, cz-s], [cx-s, cy+s, cz-s],
        [cx-s, cy-s, cz+s], [cx+s, cy-s, cz+s],
        [cx+s, cy+s, cz+s], [cx-s, cy+s, cz+s],
    ])
    edges = [
        (0,1),(1,2),(2,3),(3,0),  # back face
        (4,5),(5,6),(6,7),(7,4),  # front face
        (0,4),(1,5),(2,6),(3,7),  # connecting edges
    ]
    return verts, edges


W, H = 640, 480
fx, fy = 500.0, 500.0
cx, cy = W / 2, H / 2
K = make_intrinsic(fx, fy, cx, cy)

R = np.eye(3)
t = np.zeros(3)

verts, edges = make_cube(center=(0, 0, 5), size=2.0)
pixels, depths = project_points(verts, K, R, t)

fig, ax = plt.subplots(1, 1, figsize=(7, 5))
for i, j in edges:
    color = plt.cm.viridis(0.3 + 0.4 * (depths[i] + depths[j]) / (2 * depths.max()))
    ax.plot([pixels[i, 0], pixels[j, 0]],
            [pixels[i, 1], pixels[j, 1]],
            color=color, linewidth=2)
ax.scatter(pixels[:, 0], pixels[:, 1], c=depths, cmap="viridis", s=40, zorder=5)
ax.set_xlim(0, W)
ax.set_ylim(H, 0)
ax.set_aspect("equal")
ax.set_xlabel("u (pixels)")
ax.set_ylabel("v (pixels)")
ax.set_title("Projected 3D Cube Wireframe (depth-colored)")
plt.colorbar(ax.collections[0], ax=ax, label="Depth (Z)")
plt.tight_layout()
plt.show()

print("Original 3D points (first 4):")
print(verts[:4])
print("\nProjected 2D pixels (first 4):")
print(pixels[:4])
print("\n⚠ Notice: two different 3D points with different depths")
print("  can project to nearby pixels. Depth is LOST in projection.")

### 1.5 Projection is Lossy

The projection $\mathbb{R}^3 \to \mathbb{R}^2$ maps an entire **ray** of 3D points to a
single pixel. Every point of the form:

\[
\mathbf{P}(\lambda) = \lambda K^{-1} \tilde{\mathbf{p}}, \quad \lambda > 0
\]

where $\tilde{\mathbf{p}} = (u, v, 1)^\top$ projects to the same pixel $(u, v)$. This is
the fundamental ambiguity that makes **depth estimation** the central challenge of 3D vision.

In [ ]:
# Demonstrate the lossy nature: multiple 3D points → same pixel
pixel_target = np.array([400.0, 300.0])
ray_dir = np.linalg.inv(K) @ np.array([pixel_target[0], pixel_target[1], 1.0])

print(f"Pixel target: ({pixel_target[0]:.0f}, {pixel_target[1]:.0f})")
print(f"Ray direction in camera frame: {ray_dir}\n")
print("All these 3D points project to the SAME pixel:")
for depth in [1.0, 2.0, 5.0, 10.0, 50.0]:
    P_3d = depth * ray_dir
    reproj, _ = project_points(P_3d.reshape(1, 3), K, R, t)
    print(f"  depth={depth:5.1f}  →  P=({P_3d[0]:8.3f}, {P_3d[1]:8.3f}, {P_3d[2]:8.3f})"
          f"  →  pixel=({reproj[0,0]:.1f}, {reproj[0,1]:.1f})")

---

## Section 2: Back-Projection

### 2.1 The Inverse of Projection

Given a pixel $(u, v)$ and its **depth** $d$, we can recover the 3D point in camera
coordinates:

\[
X = \frac{(u - c_x) \cdot d}{f_x}, \qquad
Y = \frac{(v - c_y) \cdot d}{f_y}, \qquad
Z = d
\]

In matrix form:

\[
\begin{pmatrix} X \\ Y \\ Z \end{pmatrix}
= d \cdot K^{-1} \begin{pmatrix} u \\ v \\ 1 \end{pmatrix}
\]

**This is the equation that builds EVERY point cloud in 3D mapping.** Whether you're using
LiDAR, stereo depth, or monocular depth estimation, the conversion from a depth map to a
3D point cloud uses exactly this formula.

### 2.2 Back-Projection Defines a Ray

Without depth information, back-projection defines a **ray** emanating from the camera
center through the pixel:

\[
\mathbf{r}(\lambda) = \lambda K^{-1} \begin{pmatrix} u \\ v \\ 1 \end{pmatrix},
\quad \lambda > 0
\]

The 3D point lies *somewhere* on this ray. Depth estimation tells us *where*.

In [ ]:
def backproject_points(pixels, depths, K):
    """
    Back-project 2D pixels + depth to 3D points in camera frame.

    Parameters
    ----------
    pixels : (N, 2) array — pixel coordinates (u, v)
    depths : (N,) array   — depth values
    K      : (3, 3) array — intrinsic matrix

    Returns
    -------
    points_3d : (N, 3) array — 3D points in camera coordinates
    """
    fx, fy = K[0, 0], K[1, 1]
    cx, cy = K[0, 2], K[1, 2]

    X = (pixels[:, 0] - cx) * depths / fx
    Y = (pixels[:, 1] - cy) * depths / fy
    Z = depths

    return np.stack([X, Y, Z], axis=1)

In [ ]:
# Round-trip verification: project → backproject should recover original 3D points
np.random.seed(42)
P_original = np.random.uniform(-2, 2, (20, 3))
P_original[:, 2] = np.abs(P_original[:, 2]) + 3  # ensure positive depth

pixels_proj, depths_proj = project_points(P_original, K, R, t)
P_recovered = backproject_points(pixels_proj, depths_proj, K)

error = np.linalg.norm(P_original - P_recovered, axis=1)
print("Round-trip error (project → backproject):")
print(f"  Max error:  {error.max():.2e}")
print(f"  Mean error: {error.mean():.2e}")
assert error.max() < 1e-10, "Round-trip failed!"
print("\n✓ Round-trip verified: project → backproject recovers original 3D points.")

In [ ]:
def make_synthetic_depth_map(W, H):
    """
    Create a synthetic depth map with a planar background and a sphere.
    """
    u_coords, v_coords = np.meshgrid(np.arange(W), np.arange(H))

    # Planar background slanting away
    depth = 5.0 + 0.002 * (v_coords - H/2)

    # Sphere in front: center at image center, radius 80px
    du = u_coords - W / 2
    dv = v_coords - H / 2
    r_px = np.sqrt(du**2 + dv**2)
    sphere_radius_px = 80
    sphere_mask = r_px < sphere_radius_px
    sphere_depth = 3.0 - 0.5 * np.sqrt(
        np.maximum(0, 1 - (r_px / sphere_radius_px)**2)
    )
    depth[sphere_mask] = sphere_depth[sphere_mask]

    return depth.astype(np.float32)


depth_map = make_synthetic_depth_map(W, H)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(depth_map, cmap="turbo")
ax.set_title("Synthetic Depth Map")
ax.set_xlabel("u (pixels)")
ax.set_ylabel("v (pixels)")
plt.colorbar(im, ax=ax, label="Depth (m)")
plt.tight_layout()
plt.show()

In [ ]:
# Back-project depth map to 3D point cloud
stride = 4  # subsample for faster visualization
u_coords, v_coords = np.meshgrid(
    np.arange(0, W, stride), np.arange(0, H, stride)
)
u_flat = u_coords.flatten().astype(np.float64)
v_flat = v_coords.flatten().astype(np.float64)
d_flat = depth_map[v_coords.flatten(), u_coords.flatten()].astype(np.float64)

pixels_all = np.stack([u_flat, v_flat], axis=1)
point_cloud = backproject_points(pixels_all, d_flat, K)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")
sc = ax.scatter(
    point_cloud[:, 0], point_cloud[:, 1], point_cloud[:, 2],
    c=d_flat, cmap="turbo", s=0.5, alpha=0.8
)
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z (depth)")
ax.set_title("Back-Projected Point Cloud")
ax.view_init(elev=-70, azim=-90)
plt.colorbar(sc, ax=ax, label="Depth", shrink=0.6)
plt.tight_layout()
plt.show()

print(f"Point cloud shape: {point_cloud.shape}")
print(f"X range: [{point_cloud[:,0].min():.2f}, {point_cloud[:,0].max():.2f}]")
print(f"Y range: [{point_cloud[:,1].min():.2f}, {point_cloud[:,1].max():.2f}]")
print(f"Z range: [{point_cloud[:,2].min():.2f}, {point_cloud[:,2].max():.2f}]")

---

## Section 3: Lens Distortion

### 3.1 Why Distortion Matters

Real lenses are not ideal pinholes. They introduce geometric distortions that must be
corrected before any geometric reasoning (triangulation, pose estimation, etc.) is valid.

There are two main types of distortion:

### 3.2 Radial Distortion — Brown-Conrady Derivation

The Brown-Conrady model (Brown 1966, Conrady 1919) derives lens distortion from
**Seidel aberration theory**. We give the step-by-step derivation here.

**Physical setup.** In an ideal thin lens, all rays from a point source converge to
a single image point. A real lens refracts rays differently depending on their distance
$r$ from the optical axis. This introduces a radial displacement $\delta r$ between
where a ray *should* land (pinhole model) and where it *actually* lands.

**Step 1 — Symmetry argument.** A rotationally symmetric lens has no preferred
orientation in the $(x, y)$ plane. Therefore the displacement $\delta r$ can depend
only on $r = \sqrt{x_n^2 + y_n^2}$, not on the azimuthal angle. Moreover, the
displacement must be an **odd** function of $r$ (it reverses sign when the ray crosses
the axis), so $\delta r / r$ is an even function of $r$.

**Step 2 — Taylor expansion.** Expand the radial displacement as a power series in $r$:

\[
\delta r = k_1 r^3 + k_2 r^5 + k_3 r^7 + \cdots
\]

Only **odd** powers of $r$ appear (even powers would make $\delta r$ even, violating
the sign-reversal requirement). The distorted radius is:

\[
r_d = r + \delta r = r\bigl(1 + k_1 r^2 + k_2 r^4 + k_3 r^6 + \cdots\bigr)
\]

**Step 3 — Cartesian form.** The displacement is purely radial, so decomposing into
$(x, y)$ components (noting $x_n = r\cos\phi$, $y_n = r\sin\phi$):

\[
x_d = x_n \cdot \frac{r_d}{r} = x_n \left(1 + k_1 r^2 + k_2 r^4 + k_3 r^6\right)
\]
\[
y_d = y_n \left(1 + k_1 r^2 + k_2 r^4 + k_3 r^6\right)
\]

where $(x_n, y_n)$ are the **normalized** (undistorted) image coordinates
$x_n = X/Z$, $y_n = Y/Z$, and $r^2 = x_n^2 + y_n^2$.

**Sign conventions:**
- $k_1 < 0$ → **barrel distortion** (lines curve outward) — common in wide-angle lenses
- $k_1 > 0$ → **pincushion distortion** (lines curve inward) — common in telephoto lenses

### 3.3 Tangential Distortion — Conrady's Decentering Model

**Physical origin.** Tangential distortion arises when lens elements are not perfectly
aligned along the optical axis. Conrady (1919) modeled a slightly decentered thin lens
and showed, via first-order perturbation of Snell's law, that the decentering introduces
displacement components both along and perpendicular to the radial direction.

**Derivation.** Let $(p_1, p_2)$ parameterize the decentering direction. The lowest-order
Seidel terms for a decentered element at radius $r$ give (Brown 1966, Eq. 3):

\[
\delta x_{\text{tang}} = 2 p_1 x_n y_n + p_2 (r^2 + 2 x_n^2)
\]
\[
\delta y_{\text{tang}} = p_1 (r^2 + 2 y_n^2) + 2 p_2 x_n y_n
\]

These can be verified by noting: (i) the terms are quadratic in $(x_n, y_n)$, as
expected from first-order Seidel theory; (ii) the $p_2$ terms produce a displacement
along $x$ that grows as $\sim 3x_n^2 + y_n^2$ near the $x$-axis, consistent with
the "thin prism" model of a tilted lens element.

**Combined model.** Adding radial and tangential components:

\[
x_d = x_n \left(1 + k_1 r^2 + k_2 r^4 + k_3 r^6\right) + 2 p_1 x_n y_n + p_2 (r^2 + 2 x_n^2)
\]
\[
y_d = y_n \left(1 + k_1 r^2 + k_2 r^4 + k_3 r^6\right) + p_1 (r^2 + 2 y_n^2) + 2 p_2 x_n y_n
\]

The final distorted pixel coordinates are:
\[
u_d = f_x \cdot x_d + c_x, \qquad v_d = f_y \cdot y_d + c_y
\]

### 3.4 Undistortion via Newton's Method

The distortion model maps **undistorted → distorted** coordinates. To go the other
direction (undistort an observed pixel), we need to **invert** the distortion function.
Since it's nonlinear, we use iterative Newton's method:

Given distorted point $(x_d, y_d)$, find $(x_n, y_n)$ such that $D(x_n, y_n) = (x_d, y_d)$.

Initialize $(x_n^{(0)}, y_n^{(0)}) = (x_d, y_d)$ and iterate:
\[
(x_n^{(k+1)}, y_n^{(k+1)}) = (x_n^{(k)}, y_n^{(k)}) - J^{-1} \bigl(D(x_n^{(k)}, y_n^{(k)}) - (x_d, y_d)\bigr)
\]

where $J$ is the $2 \times 2$ Jacobian of the distortion function.

In [ ]:
def apply_distortion(x_n, y_n, k1, k2, k3, p1, p2):
    """
    Apply radial + tangential distortion to normalized image coordinates.

    Parameters
    ----------
    x_n, y_n : arrays of normalized (undistorted) coordinates
    k1, k2, k3 : radial distortion coefficients
    p1, p2 : tangential distortion coefficients

    Returns
    -------
    x_d, y_d : arrays of distorted normalized coordinates
    """
    r2 = x_n**2 + y_n**2
    radial = 1 + k1 * r2 + k2 * r2**2 + k3 * r2**3

    x_d = x_n * radial + 2*p1*x_n*y_n + p2*(r2 + 2*x_n**2)
    y_d = y_n * radial + p1*(r2 + 2*y_n**2) + 2*p2*x_n*y_n

    return x_d, y_d


def undistort_newton(x_d, y_d, k1, k2, k3, p1, p2, n_iters=20, tol=1e-12):
    """
    Iterative undistortion using Newton's method.
    Finds the undistorted (x_n, y_n) given distorted (x_d, y_d).
    """
    x_n = x_d.copy()
    y_n = y_d.copy()

    for _ in range(n_iters):
        x_est, y_est = apply_distortion(x_n, y_n, k1, k2, k3, p1, p2)
        err_x = x_est - x_d
        err_y = y_est - y_d

        if np.max(np.abs(err_x)) < tol and np.max(np.abs(err_y)) < tol:
            break

        # Approximate Jacobian (ignoring tangential cross-terms for stability)
        r2 = x_n**2 + y_n**2
        radial = 1 + k1 * r2 + k2 * r2**2 + k3 * r2**3
        # Simple inverse: update step
        x_n = x_n - err_x / (radial + 1e-15)
        y_n = y_n - err_y / (radial + 1e-15)

    return x_n, y_n

In [ ]:
# Visualize distortion effects on a regular grid
grid_n = 20
lin = np.linspace(-0.5, 0.5, grid_n)
gx, gy = np.meshgrid(lin, lin)
gx_flat, gy_flat = gx.flatten(), gy.flatten()

distortion_params = [
    {"label": "No distortion",  "k1":  0.0,  "k2": 0.0, "k3": 0.0},
    {"label": "Barrel (k₁=-0.4)", "k1": -0.4, "k2": 0.0, "k3": 0.0},
    {"label": "Pincushion (k₁=+0.4)", "k1": 0.4, "k2": 0.0, "k3": 0.0},
    {"label": "Strong barrel (k₁=-0.8, k₂=0.3)", "k1": -0.8, "k2": 0.3, "k3": 0.0},
]

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
for ax, params in zip(axes, distortion_params):
    xd, yd = apply_distortion(
        gx_flat, gy_flat,
        params["k1"], params["k2"], params["k3"], 0.0, 0.0
    )
    xd = xd.reshape(grid_n, grid_n)
    yd = yd.reshape(grid_n, grid_n)

    for i in range(grid_n):
        ax.plot(xd[i, :], yd[i, :], "b-", linewidth=0.7, alpha=0.6)
        ax.plot(xd[:, i], yd[:, i], "r-", linewidth=0.7, alpha=0.6)
    ax.set_aspect("equal")
    ax.set_title(params["label"], fontsize=10)
    ax.set_xlim(-0.7, 0.7)
    ax.set_ylim(-0.7, 0.7)
    ax.grid(True, alpha=0.2)

fig.suptitle("Effect of Radial Distortion on a Regular Grid", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Verify undistortion round-trip
k1, k2, k3 = -0.3, 0.1, -0.02
p1, p2 = 0.001, -0.002

test_x = np.linspace(-0.4, 0.4, 50)
test_y = np.linspace(-0.3, 0.3, 50)
tx, ty = np.meshgrid(test_x, test_y)
tx_flat, ty_flat = tx.flatten(), ty.flatten()

# Forward: undistorted → distorted
xd, yd = apply_distortion(tx_flat, ty_flat, k1, k2, k3, p1, p2)

# Inverse: distorted → undistorted
xr, yr = undistort_newton(xd, yd, k1, k2, k3, p1, p2)

err = np.sqrt((tx_flat - xr)**2 + (ty_flat - yr)**2)
print(f"Undistortion round-trip error: max={err.max():.2e}, mean={err.mean():.2e}")
print("✓ Newton's method successfully inverts the distortion model.")

---

### 3.5 Jacobian of Projection with Respect to Camera Parameters

Understanding *how sensitive* the projected pixel is to each camera parameter is critical
for calibration, uncertainty propagation, and nonlinear optimization.

#### With respect to the 3D point $\mathbf{P}_c = (X, Y, Z)^T$ in camera frame

The undistorted projection is $u = f_x X/Z + c_x$, $v = f_y Y/Z + c_y$. Differentiating:

\[
J_{\mathbf{P}} = \frac{\partial(u, v)}{\partial(X, Y, Z)}
= \begin{pmatrix}
f_x / Z & 0 & -f_x X / Z^2 \\
0 & f_y / Z & -f_y Y / Z^2
\end{pmatrix}
= \frac{1}{Z}
\begin{pmatrix}
f_x & 0 & -f_x x_n \\
0 & f_y & -f_y y_n
\end{pmatrix}
\]

where $x_n = X/Z$, $y_n = Y/Z$. This Jacobian is **rank 2** for $Z > 0$ — the lost rank
corresponds to the depth direction (projection is a rank-deficient map from 3D to 2D).

#### With respect to the intrinsic parameters $\boldsymbol{\theta}_K = (f_x, f_y, c_x, c_y)$

\[
J_K = \frac{\partial(u, v)}{\partial(f_x, f_y, c_x, c_y)}
= \begin{pmatrix}
X/Z & 0 & 1 & 0 \\
0 & Y/Z & 0 & 1
\end{pmatrix}
= \begin{pmatrix}
x_n & 0 & 1 & 0 \\
0 & y_n & 0 & 1
\end{pmatrix}
\]

**Key insight:** The sensitivity to $f_x$ is proportional to $x_n = X/Z$ — points far
from the optical axis are more sensitive to focal length errors. Points near the center
($x_n \approx 0$) are insensitive to $f_x$, which is why calibration patterns must
span the full image.

#### With respect to distortion coefficients $(k_1, k_2, p_1, p_2)$

With $r^2 = x_n^2 + y_n^2$:

\[
\frac{\partial(x_d, y_d)}{\partial(k_1, k_2, p_1, p_2)}
= \begin{pmatrix}
x_n r^2 & x_n r^4 & 2 x_n y_n & r^2 + 2x_n^2 \\
y_n r^2 & y_n r^4 & r^2 + 2y_n^2 & 2 x_n y_n
\end{pmatrix}
\]

Converting to pixel coordinates: $J_{\text{dist}} = \operatorname{diag}(f_x, f_y) \cdot J$ above.

#### Distortion function's own Jacobian $J_D$

The $2 \times 2$ Jacobian of the distortion map $(x_n, y_n) \mapsto (x_d, y_d)$ is needed
for Newton's method undistortion (Section 3.4) and for propagating gradients:

\[
J_D = \begin{pmatrix}
\dfrac{\partial x_d}{\partial x_n} & \dfrac{\partial x_d}{\partial y_n} \\[6pt]
\dfrac{\partial y_d}{\partial x_n} & \dfrac{\partial y_d}{\partial y_n}
\end{pmatrix}
\]

Let $\rho = 1 + k_1 r^2 + k_2 r^4$ (the radial factor) and
$\rho' = k_1 + 2 k_2 r^2$ (its derivative w.r.t. $r^2$). Then:

\[
\frac{\partial x_d}{\partial x_n} = \rho + 2 x_n^2 \rho' + 2 p_1 y_n + 6 p_2 x_n
\]
\[
\frac{\partial x_d}{\partial y_n} = 2 x_n y_n \rho' + 2 p_1 x_n + 2 p_2 y_n
\]
\[
\frac{\partial y_d}{\partial x_n} = 2 x_n y_n \rho' + 2 p_1 x_n + 2 p_2 y_n
\]
\[
\frac{\partial y_d}{\partial y_n} = \rho + 2 y_n^2 \rho' + 6 p_1 y_n + 2 p_2 x_n
\]

Note that $J_D$ is **symmetric in the off-diagonal entries** — a consequence of the
rotational symmetry of the radial model.

---

## Section 4: Camera Calibration (Zhang's Method)

### 4.1 The Calibration Problem

Camera calibration estimates the intrinsic matrix $K$ and distortion coefficients from
observations of a known pattern (typically a checkerboard).

### 4.2 Zhang's Method — Key Ideas

Zhang's calibration method (TPAMI 2000) works as follows:

1. **Capture** $n \geq 3$ images of a planar checkerboard at different orientations.

2. **Detect** the checkerboard corners in each image, giving 2D-3D correspondences
   $(\mathbf{m}_i, \mathbf{M}_i)$ where $\mathbf{M}_i$ lies on the $Z=0$ plane.

3. **Estimate homographies**: For each view, the relationship between the pattern plane
   and the image is a **homography** $H$ (a $3 \times 3$ matrix). Using DLT (Direct
   Linear Transform), each pair of correspondences gives two equations:

   \[
   \lambda \tilde{\mathbf{m}} = H \tilde{\mathbf{M}} = K [\mathbf{r}_1 \; \mathbf{r}_2 \; \mathbf{t}]
   \begin{pmatrix} X \\ Y \\ 1 \end{pmatrix}
   \]

   since $Z = 0$ eliminates the third column of $R$.

4. **Constrain $K$ via the image of the absolute conic (full SVD derivation)**:

   From $H = K[\mathbf{r}_1 \; \mathbf{r}_2 \; \mathbf{t}]$, the columns of $H$ satisfy
   $\mathbf{h}_i = K\mathbf{r}_i$, so $\mathbf{r}_i = K^{-1}\mathbf{h}_i$.

   The orthonormality of $R$ requires $\mathbf{r}_1^T \mathbf{r}_2 = 0$ and
   $\|\mathbf{r}_1\| = \|\mathbf{r}_2\|$:

   \[
   (K^{-1}\mathbf{h}_1)^T (K^{-1}\mathbf{h}_2) = \mathbf{h}_1^T \underbrace{K^{-T}K^{-1}}_{B}\,\mathbf{h}_2 = 0
   \]
   \[
   \mathbf{h}_1^T B \mathbf{h}_1 = \mathbf{h}_2^T B \mathbf{h}_2
   \]

   where $B = K^{-T}K^{-1}$ is the **image of the absolute conic** (IAC), a $3 \times 3$
   symmetric matrix with 6 independent entries.

   **Vectorizing the constraint.** Define the 6-vector
   $\mathbf{b} = (B_{11}, B_{12}, B_{13}, B_{22}, B_{23}, B_{33})^T$.
   For any two columns $\mathbf{h}_i, \mathbf{h}_j$ of $H$, the product
   $\mathbf{h}_i^T B \mathbf{h}_j$ is **linear** in $\mathbf{b}$:

   \[
   \mathbf{h}_i^T B \mathbf{h}_j
   = \underbrace{\begin{pmatrix}
   h_{i1}h_{j1},\;
   h_{i1}h_{j2} + h_{i2}h_{j1},\;
   h_{i1}h_{j3} + h_{i3}h_{j1},\;
   h_{i2}h_{j2},\;
   h_{i2}h_{j3} + h_{i3}h_{j2},\;
   h_{i3}h_{j3}
   \end{pmatrix}}_{\mathbf{v}_{ij}^T}
   \mathbf{b}
   \]

   Each homography gives **two equations** in $\mathbf{b}$:

   \[
   \mathbf{v}_{12}^T \mathbf{b} = 0, \qquad
   (\mathbf{v}_{11} - \mathbf{v}_{22})^T \mathbf{b} = 0
   \]

   Stacking all $n$ homographies gives a $2n \times 6$ system $V\mathbf{b} = \mathbf{0}$.
   For $n \ge 3$, this system is (over-)determined up to scale. We solve by computing the SVD
   $V = U\Sigma W^T$ and taking $\mathbf{b}$ as the **last column of $W$** (the right
   singular vector corresponding to the smallest singular value), exactly as in the DLT
   derivation of Notebook 02.

5. **Extract $K$** from $B$ via Cholesky decomposition: since $B = K^{-T}K^{-1}$ is
   symmetric positive-definite, compute $B = LL^T$ (Cholesky), giving $L = K^{-T}$ and
   thus $K = (L^{-1})^T$. Normalize so $K_{33} = 1$.

6. **Refine** all parameters jointly by minimizing reprojection error via
   Levenberg-Marquardt optimization (see Jacobian derivation below).

### 4.3 Reprojection Error

The quality of calibration is measured by the **mean reprojection error**:

\[
\epsilon_{\text{reproj}} = \frac{1}{N} \sum_{i=1}^{N}
\left\| \mathbf{m}_i^{\text{observed}} - \hat{\mathbf{m}}_i^{\text{projected}} \right\|_2
\]

Good calibrations achieve $\epsilon_{\text{reproj}} < 0.5$ pixels.

### 4.4 Reprojection Error Jacobians for Nonlinear Refinement

Zhang's method (step 6) refines all parameters by minimizing total reprojection error
via **Levenberg-Marquardt**. This requires the Jacobian of the residual vector with respect
to all optimized parameters.

#### The residual vector

For point $i$ observed in view $j$, the residual is:

\[
\mathbf{e}_{ij} = \mathbf{m}_{ij}^{\text{obs}} - \hat{\pi}(K,\; \mathbf{k},\; R_j,\; \mathbf{t}_j,\; \mathbf{M}_i)
\in \mathbb{R}^2
\]

where $\hat{\pi}$ is the full projection pipeline: world-to-camera → normalize → distort → pixelize.

#### Chain-rule decomposition

Break $\hat{\pi}$ into four stages and apply the chain rule:

\[
\underbrace{
\frac{\partial \mathbf{e}}{\partial \boldsymbol{\theta}}
}_{2 \times |\boldsymbol{\theta}|}
= -\,
\underbrace{
\frac{\partial(u, v)}{\partial(x_d, y_d)}
}_{2 \times 2}
\cdot
\underbrace{
\frac{\partial(x_d, y_d)}{\partial(x_n, y_n)}
}_{2 \times 2}
\cdot
\underbrace{
\frac{\partial(x_n, y_n)}{\partial \mathbf{P}_c}
}_{2 \times 3}
\cdot
\underbrace{
\frac{\partial \mathbf{P}_c}{\partial \boldsymbol{\theta}}
}_{3 \times |\boldsymbol{\theta}|}
\]

**Stage 4 → 3** (pixel ← distorted normalized):

\[
\frac{\partial(u, v)}{\partial(x_d, y_d)} = \begin{pmatrix} f_x & 0 \\ 0 & f_y \end{pmatrix}
\]

**Stage 3 → 2** (distorted ← undistorted normalized):

This is the $2 \times 2$ distortion Jacobian $J_D$ from Section 3.5.

**Stage 2 → 1** (normalized ← camera-frame 3D):

\[
\frac{\partial(x_n, y_n)}{\partial(X, Y, Z)}
= \frac{1}{Z}\begin{pmatrix} 1 & 0 & -x_n \\ 0 & 1 & -y_n \end{pmatrix}
\]

**Stage 1** (camera-frame ← parameters):

$\mathbf{P}_c = R(\boldsymbol{\omega}_j)\,\mathbf{M}_i + \mathbf{t}_j$, where
$\boldsymbol{\omega}_j \in \mathbb{R}^3$ is the axis-angle rotation vector.

- **w.r.t. translation** $\mathbf{t}_j$: $\;\dfrac{\partial \mathbf{P}_c}{\partial \mathbf{t}_j} = I_3$

- **w.r.t. rotation** $\boldsymbol{\omega}_j$ (axis-angle): Using the Rodrigues derivative,
  $\dfrac{\partial(R\mathbf{M})}{\partial \boldsymbol{\omega}} = -[R\mathbf{M}]_\times \cdot J_r(\boldsymbol{\omega})$
  where $[\cdot]_\times$ is the skew-symmetric (hat) operator and $J_r$ is the right Jacobian of $SO(3)$
  (derived in Notebook 04).

- **w.r.t. 3D point** $\mathbf{M}_i$: $\;\dfrac{\partial \mathbf{P}_c}{\partial \mathbf{M}_i} = R_j$

#### Assembling the full Jacobian

The optimized parameter vector for calibration is:

\[
\boldsymbol{\theta} = \bigl(\underbrace{f_x, f_y, c_x, c_y}_{\text{intrinsics}},\;
\underbrace{k_1, k_2, p_1, p_2}_{\text{distortion}},\;
\underbrace{\boldsymbol{\omega}_1, \mathbf{t}_1, \ldots, \boldsymbol{\omega}_n, \mathbf{t}_n}_{\text{extrinsics}}\bigr)
\]

The full Jacobian $J \in \mathbb{R}^{2NM \times (4 + 4 + 6n)}$ (for $N$ points observed
in $M$ views) is **sparse**: each residual $\mathbf{e}_{ij}$ depends only on the
intrinsics, distortion, and the extrinsics of view $j$. This sparsity is exploited by
the **Schur complement trick** in Levenberg-Marquardt, reducing the normal equations
from $O((4+4+6n)^3)$ to $O(n \cdot 6^3)$. This is the same sparse structure that
appears in bundle adjustment (Notebook 07).

In [ ]:
def generate_checkerboard_detections(K_true, dist_coeffs, n_views=15,
                                      board_size=(9, 6), square_size=0.025):
    """
    Generate synthetic checkerboard corner detections for camera calibration.

    Simulates n_views of a checkerboard pattern seen from different poses,
    projects the 3D corners through the camera model (with distortion),
    and adds Gaussian noise to the detected pixel locations.

    DLT math (for reference):
        For a planar target at Z=0, the projection simplifies to a homography:
            λ [u, v, 1]ᵀ = K [r₁ r₂ t] [X, Y, 1]ᵀ
        Each correspondence (Mᵢ, mᵢ) yields 2 equations. With ≥4 points
        per view, we solve for H via SVD of the DLT system Ah = 0.
    """
    obj_points_single = np.zeros((board_size[0] * board_size[1], 3), np.float32)
    obj_points_single[:, :2] = np.mgrid[
        0:board_size[0], 0:board_size[1]
    ].T.reshape(-1, 2) * square_size

    obj_points_list = []
    img_points_list = []

    rng = np.random.RandomState(123)

    for _ in range(n_views):
        rvec = rng.uniform(-0.5, 0.5, 3).astype(np.float64)
        tvec = np.array([
            rng.uniform(-0.1, 0.1),
            rng.uniform(-0.1, 0.1),
            rng.uniform(0.3, 0.6),
        ], dtype=np.float64)

        img_pts, _ = cv2.projectPoints(
            obj_points_single, rvec, tvec, K_true.astype(np.float64),
            np.array(dist_coeffs, dtype=np.float64)
        )
        img_pts = img_pts.reshape(-1, 2)

        noise = rng.normal(0, 0.3, img_pts.shape)  # sub-pixel noise
        img_pts += noise

        # Only keep views where all points are within image bounds
        if (img_pts[:, 0].min() > 10 and img_pts[:, 0].max() < W - 10 and
            img_pts[:, 1].min() > 10 and img_pts[:, 1].max() < H - 10):
            obj_points_list.append(obj_points_single.copy())
            img_points_list.append(img_pts.astype(np.float32))

    return obj_points_list, img_points_list

In [ ]:
# Ground truth camera parameters
K_true = make_intrinsic(520.0, 520.0, 320.0, 240.0)
dist_true = [-0.2, 0.1, 0.001, -0.001, 0.0]

obj_pts, img_pts = generate_checkerboard_detections(
    K_true, dist_true, n_views=20
)
print(f"Generated {len(obj_pts)} valid checkerboard views")

ret, K_est, dist_est, rvecs, tvecs = cv2.calibrateCamera(
    obj_pts, img_pts, (W, H), None, None
)

print(f"\nReprojection error: {ret:.4f} pixels")
print(f"\nEstimated K:\n{K_est}")
print(f"\nTrue K:\n{K_true}")
print(f"\nEstimated distortion: {dist_est.ravel()[:5]}")
print(f"True distortion:      {dist_true}")

print(f"\n--- Parameter Errors ---")
print(f"fx error: {abs(K_est[0,0] - K_true[0,0]):.3f} px")
print(f"fy error: {abs(K_est[1,1] - K_true[1,1]):.3f} px")
print(f"cx error: {abs(K_est[0,2] - K_true[0,2]):.3f} px")
print(f"cy error: {abs(K_est[1,2] - K_true[1,2]):.3f} px")

In [ ]:
# Visualize reprojection errors across all views
all_errors = []
for i in range(len(obj_pts)):
    reproj_pts, _ = cv2.projectPoints(
        obj_pts[i], rvecs[i], tvecs[i], K_est, dist_est
    )
    err = np.linalg.norm(
        img_pts[i] - reproj_pts.reshape(-1, 2), axis=1
    )
    all_errors.extend(err.tolist())

all_errors = np.array(all_errors)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(all_errors, bins=50, edgecolor="black", alpha=0.7)
axes[0].axvline(all_errors.mean(), color="r", linestyle="--",
                label=f"Mean = {all_errors.mean():.3f} px")
axes[0].set_xlabel("Reprojection Error (pixels)")
axes[0].set_ylabel("Count")
axes[0].set_title("Reprojection Error Distribution")
axes[0].legend()

view_idx = 0
reproj_pts, _ = cv2.projectPoints(
    obj_pts[view_idx], rvecs[view_idx], tvecs[view_idx], K_est, dist_est
)
reproj_pts = reproj_pts.reshape(-1, 2)
axes[1].scatter(img_pts[view_idx][:, 0], img_pts[view_idx][:, 1],
                marker="+", s=50, c="blue", label="Detected")
axes[1].scatter(reproj_pts[:, 0], reproj_pts[:, 1],
                marker="o", s=20, facecolors="none", edgecolors="red",
                label="Reprojected")
axes[1].set_xlim(0, W)
axes[1].set_ylim(H, 0)
axes[1].set_aspect("equal")
axes[1].set_title(f"View {view_idx}: Detected vs Reprojected")
axes[1].legend()

plt.tight_layout()
plt.show()

---

## Section 5: The Camera Matrix $P = K[R|\mathbf{t}]$

### 5.1 The $3 \times 4$ Projection Matrix

The complete camera projection matrix combines intrinsics and extrinsics into a single
$3 \times 4$ matrix:

\[
P = K [R | \mathbf{t}] \in \mathbb{R}^{3 \times 4}
\]

This matrix has **11 degrees of freedom**:
- 5 intrinsic: $f_x, f_y, c_x, c_y, s$ (skew)
- 3 rotation: parameterized by axis-angle, Euler angles, or quaternion
- 3 translation: $t_x, t_y, t_z$

(The matrix has 12 entries, but an overall scale is arbitrary in projective space,
giving $12 - 1 = 11$ DOF.)

### 5.2 Decomposition via RQ Factorization

Given $P$, we can recover $K$, $R$, and $\mathbf{t}$ using RQ decomposition of the
left $3 \times 3$ submatrix $M = P_{[:, :3]}$:

1. Compute $M = P[:, :3]$
2. RQ-decompose $M = KR$ where $K$ is upper triangular and $R$ is orthogonal
3. Extract $\mathbf{t} = K^{-1} P[:, 3]$
4. Normalize so that $K[2,2] = 1$ and diagonal entries of $K$ are positive

In [ ]:
def compute_P(K, R, t):
    """
    Compose the 3×4 camera projection matrix P = K[R|t].

    Parameters
    ----------
    K : (3, 3) — intrinsic matrix
    R : (3, 3) — rotation matrix (world → camera)
    t : (3,)   — translation vector

    Returns
    -------
    P : (3, 4) — camera projection matrix
    """
    Rt = np.hstack([R, t.reshape(3, 1)])
    return K @ Rt


def decompose_P(P):
    """
    Decompose P into K, R, t using RQ factorization.

    The left 3×3 block M = P[:,:3] is factored as M = K·R
    via QR decomposition of M⁻¹ (since RQ = (Q'R')ᵀ where
    Q'R' = QR(M⁻ᵀ)).

    Returns
    -------
    K : (3, 3) upper triangular intrinsic matrix (K[2,2]=1)
    R : (3, 3) rotation matrix (det(R) = +1)
    t : (3,)   translation vector
    """
    M = P[:, :3]

    # RQ decomposition via QR of the flipped matrix
    # M = K·R where K is upper-triangular and R is orthogonal.
    # Flip M, compute QR, flip back to get RQ.
    flip = np.array([[0, 0, 1], [0, 1, 0], [1, 0, 0]], dtype=np.float64)
    Q, R_upper = np.linalg.qr((flip @ M @ flip).T)
    K_out = (flip @ R_upper.T @ flip)   # upper-triangular (intrinsics)
    R_out = (flip @ Q.T @ flip)          # orthogonal (rotation)

    # Ensure positive diagonal in K
    D = np.diag(np.sign(np.diag(K_out)))
    K_out = K_out @ D
    R_out = D @ R_out

    # Normalize so K[2,2] = 1
    K_out = K_out / K_out[2, 2]

    # Ensure R is a proper rotation (det = +1)
    if np.linalg.det(R_out) < 0:
        K_out = -K_out
        R_out = -R_out

    t_out = np.linalg.inv(K_out) @ P[:, 3]

    return K_out, R_out, t_out

In [ ]:
K_test = make_intrinsic(800.0, 750.0, 320.0, 240.0)

# Rotation: 15° about Y axis
theta = np.radians(15)
R_test = np.array([
    [ np.cos(theta), 0, np.sin(theta)],
    [0,              1, 0             ],
    [-np.sin(theta), 0, np.cos(theta)],
])
t_test = np.array([0.5, -0.3, 2.0])

# Compose P
P_test = compute_P(K_test, R_test, t_test)
print("Original P:")
print(P_test)

# Decompose P back
K_rec, R_rec, t_rec = decompose_P(P_test)

print(f"\n--- Round-trip Verification ---")
print(f"K error (Frobenius): {np.linalg.norm(K_test - K_rec):.2e}")
print(f"R error (Frobenius): {np.linalg.norm(R_test - R_rec):.2e}")
print(f"t error (L2):        {np.linalg.norm(t_test - t_rec):.2e}")

# Recompose and compare
P_rec = compute_P(K_rec, R_rec, t_rec)
# P is defined up to scale
P_rec_scaled = P_rec * (P_test[2, 3] / P_rec[2, 3])
print(f"P recomposition error: {np.linalg.norm(P_test - P_rec_scaled):.2e}")
print("\n✓ decompose → recompose round-trip verified.")

---

## Section 6: Field of View and Viewing Frustum

### 6.1 Field of View from Intrinsics

The **horizontal** and **vertical field of view** (FOV) are determined by the focal
length and sensor size:

\[
\text{FOV}_h = 2 \arctan\!\left(\frac{W}{2 f_x}\right), \qquad
\text{FOV}_v = 2 \arctan\!\left(\frac{H}{2 f_y}\right)
\]

Shorter focal length → wider FOV → more context but more distortion.

### 6.2 The Viewing Frustum

The **viewing frustum** is the truncated pyramid that defines the volume of space visible
to the camera, bounded by the near and far clipping planes. Its four edges are the
back-projected rays from the four image corners.

In [ ]:
def compute_fov(fx, fy, W, H):
    """Compute horizontal and vertical FOV in degrees."""
    fov_h = 2 * np.degrees(np.arctan(W / (2 * fx)))
    fov_v = 2 * np.degrees(np.arctan(H / (2 * fy)))
    return fov_h, fov_v


def make_frustum_points(K, W, H, near=0.1, far=5.0):
    """Compute the 8 corners of the viewing frustum in camera frame."""
    K_inv = np.linalg.inv(K)
    corners_px = np.array([
        [0, 0, 1], [W, 0, 1], [W, H, 1], [0, H, 1]
    ], dtype=np.float64)

    rays = (K_inv @ corners_px.T).T  # (4, 3) ray directions
    rays = rays / rays[:, 2:3]       # normalize so Z=1

    near_pts = rays * near
    far_pts = rays * far

    return near_pts, far_pts


# Compare FOV for different focal lengths
focal_lengths = [200, 400, 600, 800, 1200]
print(f"{'fx':>6s}  {'FOV_h':>8s}  {'FOV_v':>8s}")
print("-" * 28)
for fl in focal_lengths:
    fov_h, fov_v = compute_fov(fl, fl, W, H)
    print(f"{fl:6d}  {fov_h:7.1f}°  {fov_v:7.1f}°")

In [ ]:
# Visualize viewing frustums for different focal lengths
fig = plt.figure(figsize=(14, 6))

frustum_focals = [250, 500, 1000]
colors = ["#e74c3c", "#3498db", "#2ecc71"]

ax = fig.add_subplot(111, projection="3d")

for fl, color in zip(frustum_focals, colors):
    K_f = make_intrinsic(fl, fl, W/2, H/2)
    near_pts, far_pts = make_frustum_points(K_f, W, H, near=0.5, far=4.0)

    fov_h, _ = compute_fov(fl, fl, W, H)

    # Draw frustum edges
    origin = np.zeros(3)
    for i in range(4):
        ax.plot3D(*zip(origin, far_pts[i]), color=color, alpha=0.4, linewidth=1)

    # Draw near and far rectangles
    for pts in [near_pts, far_pts]:
        rect = np.vstack([pts, pts[0]])
        ax.plot3D(rect[:, 0], rect[:, 1], rect[:, 2],
                  color=color, linewidth=2, label=f"f={fl}px (FOV={fov_h:.0f}°)")

    # Draw side faces as transparent polygons
    for i in range(4):
        j = (i + 1) % 4
        verts_face = [
            [near_pts[i], near_pts[j], far_pts[j], far_pts[i]]
        ]
        face = Poly3DCollection(verts_face, alpha=0.05, facecolor=color)
        ax.add_collection3d(face)

# Camera origin
ax.scatter([0], [0], [0], color="black", s=60, zorder=10)
ax.text(0, 0, -0.3, "Camera", ha="center", fontsize=9)

ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z (depth)")
ax.set_title("Viewing Frustums for Different Focal Lengths")

handles, labels = ax.get_legend_handles_labels()
unique = dict(zip(labels, handles))
ax.legend(unique.values(), unique.keys(), loc="upper left")

ax.view_init(elev=20, azim=-60)
plt.tight_layout()
plt.show()

---

## Section 7: Beyond Pinhole — Fisheye Camera Models

### 7.1 When the Pinhole Model Breaks

The standard radial distortion model (polynomial in $r^2$) becomes numerically unstable
for FOV $> 120°$ and cannot represent FOV $\geq 180°$ at all, since
$\tan(\theta) \to \infty$ as $\theta \to 90°$.

Fisheye lenses intentionally violate the pinhole assumption to capture ultra-wide FOV.
They require dedicated projection models.

### 7.2 Kannala-Brandt Model

The Kannala-Brandt model (TPAMI 2006) uses the **angle from the optical axis** $\theta$
rather than $\tan\theta$:

\[
\theta_d = \theta \left(1 + k_1 \theta^2 + k_2 \theta^4 + k_3 \theta^6 + k_4 \theta^8\right)
\]

where $\theta = \text{atan2}\!\left(\sqrt{X^2 + Y^2},\; Z\right)$ is the incidence angle (using `atan2`
rather than `arctan` to handle $\theta \ge \pi/2$, i.e. FOV $\ge$ 180°). The
projected point is then:

\[
u = f_x \cdot \theta_d \cdot \frac{x_n}{r} + c_x, \qquad
v = f_y \cdot \theta_d \cdot \frac{y_n}{r} + c_y
\]

where $r = \sqrt{x_n^2 + y_n^2}$.

**Key advantage**: $\theta$ is bounded, so this naturally handles FOV $> 180°$.

### 7.3 Double Sphere Model

The **Double Sphere** model (Usenko et al., 3DV 2018) offers a closed-form projection
and unprojection with only **2 distortion parameters** $(\alpha, \xi)$:

\[
d_1 = \sqrt{X^2 + Y^2 + Z^2}, \qquad
d_2 = \sqrt{X^2 + Y^2 + (\xi d_1 + Z)^2}
\]

\[
u = f_x \frac{X}{\alpha d_2 + (1 - \alpha)(\xi d_1 + Z)} + c_x
\]
\[
v = f_y \frac{Y}{\alpha d_2 + (1 - \alpha)(\xi d_1 + Z)} + c_y
\]

**Step-by-step derivation.** The model composes two sphere projections with a pinhole step.

**Step 1 — First sphere projection.** A 3D point $\mathbf{P} = (X, Y, Z)^\top$ is
projected onto the unit sphere centered at the origin. The projected point lies at
$\mathbf{P}/d_1$ where:

$$
d_1 = \|\mathbf{P}\| = \sqrt{X^2 + Y^2 + Z^2}
$$

**Step 2 — Axial shift by $\xi$.** The point on the first sphere is shifted by $\xi$
along the optical axis $Z$, modelling a lens element offset from the pinhole center.
In unnormalized coordinates (scaling back by $d_1$) the shifted point is:

$$
\mathbf{Q} = \begin{pmatrix} X \\ Y \\ \xi\, d_1 + Z \end{pmatrix}
$$

**Step 3 — Second sphere projection.** Project $\mathbf{Q}$ onto a second unit sphere:

$$
d_2 = \|\mathbf{Q}\| = \sqrt{X^2 + Y^2 + (\xi\, d_1 + Z)^2}
$$

A standard pinhole projection of the second-sphere point $\mathbf{Q}/d_2$ would
divide $(X, Y)$ by the effective depth $d_2$. A pure first-sphere pinhole
(without the second sphere) would divide by $(\xi\, d_1 + Z)$.

**Step 4 — Blending with $\alpha$.** The parameter $\alpha \in [0, 1]$ interpolates
between these two regimes. Define the effective denominator:

$$
w = \alpha\, d_2 + (1 - \alpha)(\xi\, d_1 + Z)
$$

When $\alpha = 0$, we get a shifted-sphere pinhole; when $\alpha = 1$, a pure
second-sphere projection. The final pixel coordinates are:

$$
u = f_x \frac{X}{w} + c_x, \qquad v = f_y \frac{Y}{w} + c_y
$$

**Validity condition.** A point is projectable if and only if $w > 0$, which requires
the point to be in front of both virtual spheres. For $\alpha > 0.5$, the valid
field of view exceeds $180°$, making this model suitable for fisheye lenses.

**Why only 2 distortion parameters?** Classical models (Brown-Conrady, Kannala-Brandt)
use polynomial series and need 4–6 coefficients. The Double Sphere model instead
encodes the lens geometry directly via the two physical parameters $(\xi, \alpha)$,
yielding comparable accuracy on fisheye lenses with a much more compact and
numerically stable representation (Usenko et al. 2018).

### 7.4 Real-World Applications

| System | Camera Type | FOV | Model |
|--------|------------|-----|-------|
| Skydio X10 | 6 navigation cameras | ~200° | Fisheye |
| Meta Quest 3 | 4 tracking cameras | ~150° | Fisheye |
| iPhone LiDAR | Standard lens | ~60° | Pinhole |
| Intel RealSense D435 | Stereo IR | ~87° | Pinhole + Brown-Conrady |

### 7.5 DUSt3R/MASt3R: Sidestepping Camera Models

Modern neural approaches like **DUSt3R** (CVPR 2024) and **MASt3R** take a radically
different approach: instead of explicitly modeling camera intrinsics and distortion, they
directly predict **3D pointmaps** from image pairs using a transformer architecture.

This means they implicitly learn the camera model from data, making them camera-agnostic.
However, explicit camera models remain essential for:
- Real-time applications (SLAM, visual odometry)
- Metric accuracy (the neural predictions are up to scale)
- Sensor fusion (combining cameras with IMU, LiDAR)

In [ ]:
def kannala_brandt_project(P_cam, fx, fy, cx, cy, k1, k2, k3, k4):
    """
    Kannala-Brandt fisheye projection.

    Projects 3D points in camera frame to pixel coordinates using
    the equidistant fisheye model with polynomial correction.
    """
    X, Y, Z = P_cam[:, 0], P_cam[:, 1], P_cam[:, 2]
    r = np.sqrt(X**2 + Y**2)
    theta = np.arctan2(r, Z)

    theta2 = theta * theta
    theta_d = theta * (1 + k1*theta2 + k2*theta2**2 +
                       k3*theta2**3 + k4*theta2**4)

    # Avoid division by zero for points on the optical axis
    scale = np.where(r > 1e-8, theta_d / r, np.ones_like(r))

    u = fx * scale * X + cx
    v = fy * scale * Y + cy
    return np.stack([u, v], axis=1)


def double_sphere_project(P_cam, fx, fy, cx, cy, alpha, xi):
    """
    Double Sphere camera model projection.

    Two parameters (alpha, xi) control the distortion.
    """
    X, Y, Z = P_cam[:, 0], P_cam[:, 1], P_cam[:, 2]

    d1 = np.sqrt(X**2 + Y**2 + Z**2)
    d2 = np.sqrt(X**2 + Y**2 + (xi * d1 + Z)**2)

    denom = alpha * d2 + (1 - alpha) * (xi * d1 + Z)

    # Validity check: point must be in front of both spheres
    valid = denom > 1e-8

    u = np.where(valid, fx * X / denom + cx, np.nan)
    v = np.where(valid, fy * Y / denom + cy, np.nan)

    return np.stack([u, v], axis=1)

In [ ]:
# Compare pinhole vs fisheye projections
n_pts = 5000
rng = np.random.RandomState(0)
phi = rng.uniform(0, 2 * np.pi, n_pts)     # azimuth
cos_theta = rng.uniform(0.05, 1.0, n_pts)   # elevation cosine (avoid optical axis)
sin_theta = np.sqrt(1 - cos_theta**2)

dist_3d = 5.0
P_hemi = dist_3d * np.stack([
    sin_theta * np.cos(phi),
    sin_theta * np.sin(phi),
    cos_theta
], axis=1)

W_fish, H_fish = 640, 640
cx_f, cy_f = W_fish / 2, H_fish / 2
f_fish = 200.0

# Pinhole projection
K_pinhole = make_intrinsic(f_fish, f_fish, cx_f, cy_f)
pix_pinhole, _ = project_points(P_hemi, K_pinhole, np.eye(3), np.zeros(3))

# Kannala-Brandt projection
pix_kb = kannala_brandt_project(P_hemi, f_fish, f_fish, cx_f, cy_f,
                                 k1=-0.05, k2=0.01, k3=0.0, k4=0.0)

# Double Sphere projection
pix_ds = double_sphere_project(P_hemi, f_fish, f_fish, cx_f, cy_f,
                                alpha=0.6, xi=1.2)

theta_color = np.degrees(np.arctan2(
    np.sqrt(P_hemi[:, 0]**2 + P_hemi[:, 1]**2), P_hemi[:, 2]
))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
titles = ["Pinhole", "Kannala-Brandt", "Double Sphere"]
pixel_sets = [pix_pinhole, pix_kb, pix_ds]

for ax, pix, title in zip(axes, pixel_sets, titles):
    valid = (
        np.isfinite(pix[:, 0]) & np.isfinite(pix[:, 1]) &
        (pix[:, 0] > -W_fish) & (pix[:, 0] < 2*W_fish) &
        (pix[:, 1] > -H_fish) & (pix[:, 1] < 2*H_fish)
    )
    ax.scatter(pix[valid, 0], pix[valid, 1], c=theta_color[valid],
               cmap="plasma", s=1, alpha=0.5)
    ax.set_xlim(-50, W_fish + 50)
    ax.set_ylim(H_fish + 50, -50)
    ax.set_aspect("equal")
    ax.set_title(title)
    ax.add_patch(plt.Rectangle((0, 0), W_fish, H_fish,
                               fill=False, edgecolor="gray", linewidth=1))

fig.suptitle(
    "Projection of Hemispherical Points: Pinhole vs Fisheye Models\n"
    "(color = incidence angle θ from optical axis)",
    fontsize=12
)
plt.tight_layout()
plt.show()

### Exercise (a): Project a 3D Cube with Different Focal Lengths

Project the same 3D cube onto virtual cameras with $f_x \in \{200, 500, 1000\}$ pixels.
Observe how focal length affects the apparent size and perspective distortion.

In [ ]:
verts_ex, edges_ex = make_cube(center=(0, 0, 6), size=2.0)
focal_lengths_ex = [200, 500, 1000]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, fl in zip(axes, focal_lengths_ex):
    K_ex = make_intrinsic(fl, fl, W/2, H/2)
    pix, dep = project_points(verts_ex, K_ex, np.eye(3), np.zeros(3))

    for i, j in edges_ex:
        ax.plot([pix[i, 0], pix[j, 0]], [pix[i, 1], pix[j, 1]],
                "b-", linewidth=1.5)
    ax.scatter(pix[:, 0], pix[:, 1], c="red", s=20, zorder=5)
    ax.set_xlim(0, W)
    ax.set_ylim(H, 0)
    ax.set_aspect("equal")
    fov_h, _ = compute_fov(fl, fl, W, H)
    ax.set_title(f"fx = {fl} px (FOV = {fov_h:.0f}°)")
    ax.set_xlabel("u")
    ax.set_ylabel("v")

fig.suptitle("Exercise (a): Same Cube, Different Focal Lengths", fontsize=13)
plt.tight_layout()
plt.show()

### Exercise (b): Back-Project a Depth Image to Point Cloud

Generate a synthetic depth map with multiple objects, back-project to a 3D point cloud,
and visualize in matplotlib's 3D viewer.

In [ ]:
def make_multi_object_depth(W, H):
    """Depth map with a ground plane, box, and cylinder."""
    u, v = np.meshgrid(np.arange(W), np.arange(H))
    depth = np.full((H, W), 8.0, dtype=np.float32)

    # Ground plane (bottom half, receding)
    ground = v > H * 0.5
    depth[ground] = 4.0 + 0.01 * (v[ground] - H * 0.5)

    # Box: rectangle in image space
    box_mask = (u > 100) & (u < 250) & (v > 200) & (v < 380)
    depth[box_mask] = 3.5

    # Cylinder
    cyl_cx, cyl_cy, cyl_r = 450, 300, 60
    cyl_dist = np.sqrt((u - cyl_cx)**2 + (v - cyl_cy)**2)
    cyl_mask = cyl_dist < cyl_r
    depth[cyl_mask] = 3.0 - 0.5 * np.sqrt(
        np.maximum(0, 1 - (cyl_dist[cyl_mask] / cyl_r)**2)
    )

    return depth


K_ex_b = make_intrinsic(500, 500, W/2, H/2)
depth_ex = make_multi_object_depth(W, H)

stride_b = 3
ub, vb = np.meshgrid(
    np.arange(0, W, stride_b), np.arange(0, H, stride_b)
)
pixels_b = np.stack([ub.flatten().astype(float),
                     vb.flatten().astype(float)], axis=1)
depths_b = depth_ex[vb.flatten(), ub.flatten()].astype(float)

pc_b = backproject_points(pixels_b, depths_b, K_ex_b)

fig = plt.figure(figsize=(12, 5))

ax1 = fig.add_subplot(121)
ax1.imshow(depth_ex, cmap="turbo")
ax1.set_title("Depth Map")

ax2 = fig.add_subplot(122, projection="3d")
ax2.scatter(pc_b[:, 0], pc_b[:, 1], pc_b[:, 2],
            c=depths_b, cmap="turbo", s=0.3, alpha=0.7)
ax2.set_xlabel("X")
ax2.set_ylabel("Y")
ax2.set_zlabel("Z")
ax2.set_title("Back-Projected Point Cloud")
ax2.view_init(elev=-60, azim=-90)

fig.suptitle("Exercise (b): Depth Map → Point Cloud", fontsize=13)
plt.tight_layout()
plt.show()

### Exercise (c): Visualize Radial Distortion

Create a pixel grid, apply radial distortion, and show the barrel effect. Compare
with the original undistorted grid.

In [ ]:
K_ex_c = make_intrinsic(400, 400, W/2, H/2)
grid_spacing = 30

u_lines = np.arange(0, W + 1, grid_spacing)
v_lines = np.arange(0, H + 1, grid_spacing)

k1_barrel = -0.35
k2_barrel = 0.15

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (k1_val, title) in zip(axes, [(0.0, "Undistorted Grid"),
                                       (k1_barrel, f"Barrel Distortion (k₁={k1_barrel})")]):
    # Horizontal lines
    for v_val in v_lines:
        u_pts = np.linspace(0, W, 200)
        v_pts = np.full_like(u_pts, v_val)
        xn = (u_pts - K_ex_c[0, 2]) / K_ex_c[0, 0]
        yn = (v_pts - K_ex_c[1, 2]) / K_ex_c[1, 1]
        xd, yd = apply_distortion(xn, yn, k1_val, k2_barrel if k1_val != 0 else 0,
                                   0, 0, 0)
        ud = xd * K_ex_c[0, 0] + K_ex_c[0, 2]
        vd = yd * K_ex_c[1, 1] + K_ex_c[1, 2]
        ax.plot(ud, vd, "b-", linewidth=0.6, alpha=0.7)

    # Vertical lines
    for u_val in u_lines:
        v_pts = np.linspace(0, H, 200)
        u_pts = np.full_like(v_pts, u_val)
        xn = (u_pts - K_ex_c[0, 2]) / K_ex_c[0, 0]
        yn = (v_pts - K_ex_c[1, 2]) / K_ex_c[1, 1]
        xd, yd = apply_distortion(xn, yn, k1_val, k2_barrel if k1_val != 0 else 0,
                                   0, 0, 0)
        ud = xd * K_ex_c[0, 0] + K_ex_c[0, 2]
        vd = yd * K_ex_c[1, 1] + K_ex_c[1, 2]
        ax.plot(ud, vd, "r-", linewidth=0.6, alpha=0.7)

    ax.set_xlim(-50, W + 50)
    ax.set_ylim(H + 50, -50)
    ax.set_aspect("equal")
    ax.set_title(title)

fig.suptitle("Exercise (c): Radial Distortion Effect on Pixel Grid", fontsize=13)
plt.tight_layout()
plt.show()

### Exercise (d): Calibrate Camera from Synthetic Checkerboard Images

Use the calibration pipeline from Section 4 with a different set of ground-truth parameters.

In [ ]:
K_ex_d = make_intrinsic(600.0, 580.0, 310.0, 250.0)
dist_ex_d = [-0.15, 0.05, 0.0005, -0.0003, 0.0]

obj_d, img_d = generate_checkerboard_detections(
    K_ex_d, dist_ex_d, n_views=25, board_size=(8, 6), square_size=0.03
)
print(f"Views used: {len(obj_d)}")

ret_d, K_est_d, dist_est_d, rvecs_d, tvecs_d = cv2.calibrateCamera(
    obj_d, img_d, (W, H), None, None
)

print(f"\nReprojection error: {ret_d:.4f} pixels")
print(f"\n{'Parameter':<8} {'True':>10} {'Estimated':>10} {'Error':>10}")
print("-" * 42)
for name, true_val, est_val in [
    ("fx", K_ex_d[0,0], K_est_d[0,0]),
    ("fy", K_ex_d[1,1], K_est_d[1,1]),
    ("cx", K_ex_d[0,2], K_est_d[0,2]),
    ("cy", K_ex_d[1,2], K_est_d[1,2]),
]:
    print(f"{name:<8} {true_val:10.2f} {est_val:10.2f} {abs(true_val-est_val):10.4f}")

print(f"\n✓ Calibration successful with reprojection error = {ret_d:.4f} px")

### Exercise (e): Decompose and Recompose a Camera Matrix

Given a $3 \times 4$ projection matrix $P$, extract $K$, $R$, $\mathbf{t}$ and verify
that recomposition gives back the same $P$ (up to scale).

In [ ]:
K_ex_e = make_intrinsic(700.0, 720.0, 325.0, 235.0)

# Rotation: 20° about Z, 10° about X
tz = np.radians(20)
tx = np.radians(10)
Rz = np.array([[np.cos(tz), -np.sin(tz), 0],
                [np.sin(tz),  np.cos(tz), 0],
                [0, 0, 1]])
Rx = np.array([[1, 0, 0],
                [0, np.cos(tx), -np.sin(tx)],
                [0, np.sin(tx),  np.cos(tx)]])
R_ex_e = Rz @ Rx
t_ex_e = np.array([1.0, -0.5, 3.0])

P_ex_e = compute_P(K_ex_e, R_ex_e, t_ex_e)
print("P =")
print(P_ex_e)

K_dec, R_dec, t_dec = decompose_P(P_ex_e)
print(f"\nDecomposed K:\n{K_dec}")
print(f"\nDecomposed R:\n{R_dec}")
print(f"\nDecomposed t: {t_dec}")

P_recomp = compute_P(K_dec, R_dec, t_dec)
scale = P_ex_e.ravel()[np.argmax(np.abs(P_ex_e.ravel()))] / \
        P_recomp.ravel()[np.argmax(np.abs(P_ex_e.ravel()))]
P_recomp_scaled = P_recomp * scale
err_P = np.linalg.norm(P_ex_e - P_recomp_scaled)
print(f"\nRecomposition error: {err_P:.2e}")
print(f"det(R) = {np.linalg.det(R_dec):.6f} (should be +1)")

### Exercise (f): Compute and Visualize FOV for Different Focal Lengths

Plot how the FOV changes as a function of focal length, and visualize
the corresponding frustums.

In [ ]:
fl_range = np.linspace(100, 2000, 200)
fov_h_arr = 2 * np.degrees(np.arctan(W / (2 * fl_range)))
fov_v_arr = 2 * np.degrees(np.arctan(H / (2 * fl_range)))
fov_diag_arr = 2 * np.degrees(np.arctan(
    np.sqrt(W**2 + H**2) / (2 * fl_range)
))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# FOV vs focal length
axes[0].plot(fl_range, fov_h_arr, label="Horizontal FOV", linewidth=2)
axes[0].plot(fl_range, fov_v_arr, label="Vertical FOV", linewidth=2)
axes[0].plot(fl_range, fov_diag_arr, label="Diagonal FOV", linewidth=2,
             linestyle="--")
axes[0].set_xlabel("Focal Length (pixels)")
axes[0].set_ylabel("Field of View (degrees)")
axes[0].set_title("FOV vs Focal Length")
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim(100, 2000)
axes[0].set_ylim(0, 150)

# Reference lines for common cameras
for fl_ref, name in [(280, "GoPro"), (525, "iPhone"), (1200, "Telephoto")]:
    fov_ref = 2 * np.degrees(np.arctan(W / (2 * fl_ref)))
    axes[0].axvline(fl_ref, color="gray", linestyle=":", alpha=0.5)
    axes[0].annotate(f"{name}\n{fov_ref:.0f}°", (fl_ref, fov_ref + 8),
                     ha="center", fontsize=8)

# Frustums side by side (top-down view)
ax2 = axes[1]
frustum_fls = [200, 400, 800, 1500]
cmap = plt.cm.tab10
for idx, fl in enumerate(frustum_fls):
    half_angle = np.arctan(W / (2 * fl))
    far = 6.0
    x_left = -far * np.tan(half_angle)
    x_right = far * np.tan(half_angle)

    ax2.fill(
        [0, x_left, x_right], [0, far, far],
        alpha=0.15, color=cmap(idx)
    )
    ax2.plot([0, x_left], [0, far], color=cmap(idx), linewidth=1.5)
    ax2.plot([0, x_right], [0, far], color=cmap(idx), linewidth=1.5)

    fov_val = 2 * np.degrees(half_angle)
    ax2.text(x_right + 0.1, far, f"f={fl}\n({fov_val:.0f}°)",
             fontsize=8, color=cmap(idx), va="center")

ax2.scatter([0], [0], c="black", s=60, zorder=10)
ax2.set_xlabel("X (meters)")
ax2.set_ylabel("Z — Depth (meters)")
ax2.set_title("Top-Down View of Viewing Frustums")
ax2.set_aspect("equal")
ax2.grid(True, alpha=0.2)

fig.suptitle("Exercise (f): FOV Analysis", fontsize=13)
plt.tight_layout()
plt.show()

---

## Section 7.5: Camera Matrix from Vanishing Points

*Connects to NB02 (projective geometry) and Section 4 (calibration).*

### The Image of the Absolute Conic (IAC)

The **absolute conic** $\Omega_\infty$ lives on the plane at infinity and is invariant
under Euclidean transformations. Its image under a camera with intrinsics $K$ is the
**Image of the Absolute Conic**:

$$
\omega = (KK^\top)^{-1} = K^{-\top} K^{-1}.
$$

$\omega$ is a $3 \times 3$ symmetric positive-definite matrix encoding **only intrinsics** —
it is independent of the camera pose $(R, \mathbf{t})$.

### Orthogonal Vanishing Points Constrain $\omega$

A **vanishing point** $\mathbf{v}_i$ is the image of a point at infinity along world
direction $\mathbf{d}_i$. For a camera with rotation $R$:

$$
\mathbf{v}_i \sim K R\,\mathbf{d}_i.
$$

Two directions $\mathbf{d}_i \perp \mathbf{d}_j$ in the world if and only if their
vanishing points satisfy:

$$
\mathbf{v}_i^\top \,\omega\, \mathbf{v}_j = 0.
$$

**Proof.** Writing $\mathbf{v}_i = K R \mathbf{d}_i$ (up to scale):

$$
\mathbf{v}_i^\top \omega\, \mathbf{v}_j
= (K R \mathbf{d}_i)^\top (KK^\top)^{-1} (K R \mathbf{d}_j)
= \mathbf{d}_i^\top R^\top \underbrace{K^\top K^{-\top}}_{I} \underbrace{K^{-1} K}_{I} R\,\mathbf{d}_j
= \mathbf{d}_i^\top \mathbf{d}_j. \quad \square
$$

### Solving for $K$ from Three Orthogonal VP Pairs

Three mutually orthogonal vanishing points $\mathbf{v}_1, \mathbf{v}_2, \mathbf{v}_3$ yield
three linear constraints on the six independent entries of the symmetric matrix $\omega$:

$$
\mathbf{v}_1^\top \omega\, \mathbf{v}_2 = 0, \quad
\mathbf{v}_1^\top \omega\, \mathbf{v}_3 = 0, \quad
\mathbf{v}_2^\top \omega\, \mathbf{v}_3 = 0.
$$

Since each $\mathbf{v}_i^\top \omega\, \mathbf{v}_j$ is linear in the entries of
$\omega$, we can write each constraint as a row of a linear system
$A\mathbf{w} = \mathbf{0}$ where $\mathbf{w}$ collects the 6 independent entries
$(\omega_{11}, \omega_{12}, \omega_{13}, \omega_{22}, \omega_{23}, \omega_{33})$.

Three equations are not enough for 6 unknowns (up to scale), so we add assumptions:

| Assumption | Constraint | Equation |
|:-----------|:-----------|:---------|
| Zero skew | $\omega_{12} = 0$ | $s = 0$ in $K$ |
| Known $c_x$ | $\omega_{13} + c_x\,\omega_{11} = 0$ | principal point $x$ |
| Known $c_y$ | $\omega_{23} + c_y\,\omega_{22} = 0$ | principal point $y$ |

This gives a $6 \times 6$ homogeneous system with a one-dimensional null space (the scale
of $\omega$), solvable via SVD. Finally, $K$ is extracted from
$\omega = K^{-\top}K^{-1}$ by Cholesky decomposition:

$$
\text{Cholesky}(\omega) = LL^\top \quad \Rightarrow \quad L = K^{-\top} \quad \Rightarrow \quad K = (L^{-1})^\top.
$$

In [ ]:
def rodrigues(axis, angle):
    """Rotation matrix from axis-angle via Rodrigues' formula:
    R = I + sin(θ)·[k]× + (1 - cos(θ))·[k]×²
    """
    k = np.asarray(axis, dtype=np.float64)
    k = k / np.linalg.norm(k)
    K_skew = np.array([[0, -k[2], k[1]],
                        [k[2], 0, -k[0]],
                        [-k[1], k[0], 0]])
    return np.eye(3) + np.sin(angle) * K_skew + (1 - np.cos(angle)) * K_skew @ K_skew


np.random.seed(42)

# --- Ground-truth camera ---
K_vp_true = make_intrinsic(520.0, 480.0, 320.0, 240.0)
omega_true = np.linalg.inv(K_vp_true @ K_vp_true.T)

# Three orthogonal world directions (e.g., building edges: X, Y, Z)
d1 = np.array([1.0, 0.0, 0.0])
d2 = np.array([0.0, 1.0, 0.0])
d3 = np.array([0.0, 0.0, 1.0])

# Arbitrary camera rotation
R_cam = rodrigues(np.array([0.3, -0.2, 0.5]), 0.8)

# Vanishing points: v_i = K R d_i (images of points at infinity)
v1 = K_vp_true @ R_cam @ d1
v2 = K_vp_true @ R_cam @ d2
v3 = K_vp_true @ R_cam @ d3

print("Vanishing points (homogeneous):")
for i, v in enumerate([v1, v2, v3], 1):
    px = v[:2] / v[2]
    print(f"  v{i} = [{v[0]:9.2f}, {v[1]:9.2f}, {v[2]:7.4f}]  →  pixel ({px[0]:.1f}, {px[1]:.1f})")

print(f"\nOrthogonality through IAC (should be ≈ 0):")
print(f"  v1ᵀ ω v2 = {v1 @ omega_true @ v2:.2e}")
print(f"  v1ᵀ ω v3 = {v1 @ omega_true @ v3:.2e}")
print(f"  v2ᵀ ω v3 = {v2 @ omega_true @ v3:.2e}")

# --- Set up linear system Aw = 0 for the 6 entries of symmetric omega ---
def vp_constraint_row(vi, vj):
    """Linear constraint v_i^T omega v_j = 0 on entries [w11, w12, w13, w22, w23, w33]."""
    return np.array([
        vi[0]*vj[0],                       # w11
        vi[0]*vj[1] + vi[1]*vj[0],         # w12 (appears twice by symmetry)
        vi[0]*vj[2] + vi[2]*vj[0],         # w13
        vi[1]*vj[1],                        # w22
        vi[1]*vj[2] + vi[2]*vj[1],         # w23
        vi[2]*vj[2],                        # w33
    ])

cx_known, cy_known = 320.0, 240.0

A = np.zeros((6, 6))
A[0] = vp_constraint_row(v1, v2)              # v1 ⊥ v2
A[1] = vp_constraint_row(v1, v3)              # v1 ⊥ v3
A[2] = vp_constraint_row(v2, v3)              # v2 ⊥ v3
A[3] = [0, 1, 0, 0, 0, 0]                    # zero skew: w12 = 0
A[4] = [cx_known, 0, 1, 0, 0, 0]             # w13 + cx·w11 = 0
A[5] = [0, 0, 0, cy_known, 1, 0]             # w23 + cy·w22 = 0

# Solve via SVD (null space of A)
_, S_svd, Vt = np.linalg.svd(A)
w = Vt[-1]

omega_est = np.array([
    [w[0], w[1], w[2]],
    [w[1], w[3], w[4]],
    [w[2], w[4], w[5]],
])
if omega_est[0, 0] < 0:
    omega_est = -omega_est

# Extract K via Cholesky: omega = K^{-T} K^{-1}  →  Chol(omega) = L = K^{-T}  →  K = L^{-T}
L = np.linalg.cholesky(omega_est)
K_vp_est = np.linalg.inv(L).T
K_vp_est = K_vp_est / K_vp_est[2, 2]
D = np.diag(np.sign(np.diag(K_vp_est)))
K_vp_est = D @ K_vp_est

print(f"\n--- Recovered K from 3 Orthogonal Vanishing Points ---")
print(K_vp_est.round(4))
print(f"\nTrue K:")
print(K_vp_true)
print(f"\nParameter errors:")
for name, est, true in [("fx", K_vp_est[0,0], K_vp_true[0,0]),
                         ("fy", K_vp_est[1,1], K_vp_true[1,1]),
                         ("cx", K_vp_est[0,2], K_vp_true[0,2]),
                         ("cy", K_vp_est[1,2], K_vp_true[1,2])]:
    print(f"  {name}: {abs(est - true):.6f} px")

# Verify: recovered omega should satisfy all VP constraints
for (i, j), (vi, vj) in [((1,2),(v1,v2)), ((1,3),(v1,v3)), ((2,3),(v2,v3))]:
    val = vi @ omega_est @ vj
    print(f"  v{i}ᵀ ω_est v{j} = {val:.2e}")

print("\n✓ Single-image calibration from vanishing points verified.")

---

## Summary

### What We Covered

| Concept | Key Equation | Used In |
|:--------|:------------|:--------|
| Projection | $\mathbf{p} = K[R\mid\mathbf{t}]\mathbf{P}$ | Every vision algorithm |
| Back-projection | $\mathbf{P} = d \cdot K^{-1}\tilde{\mathbf{p}}$ | Point cloud generation (NB 12) |
| Radial distortion | $x_d = x_n(1+k_1 r^2+k_2 r^4)$ | Calibration, undistortion |
| Zhang's calibration | Homography → IAC → Cholesky | Camera setup |
| Matrix decomposition | $P = K[R\mid\mathbf{t}]$ via RQ | SfM (NB 07) |
| FOV | $2\arctan(W/2f_x)$ | Sensor selection |
| Fisheye models | Kannala-Brandt, Double Sphere | Wide-angle cameras |
| VP calibration | $\mathbf{v}_i^\top\omega\,\mathbf{v}_j = 0$, $\omega = (KK^\top)^{-1}$ | Single-image calibration (NB 02) |

### Key Takeaways

1. **Projection is lossy** — it destroys depth. Recovering depth is the central problem of 3D vision.
2. **Back-projection + depth = point cloud** — this single equation powers all 3D reconstruction.
3. **Lens distortion must be corrected** before any geometric computation (epipolar geometry,
   triangulation, pose estimation).
4. **Camera calibration** estimates intrinsics from a known pattern — Zhang's method is the
   gold standard.
5. **The camera matrix** $P = K[R|\mathbf{t}]$ has 11 DOF — 5 intrinsic + 6 extrinsic.
6. **Fisheye models** are essential for wide-angle cameras (drones, AR headsets).
7. **Neural methods** (DUSt3R) can sidestep explicit camera models but sacrifice speed and
   metric accuracy.

**Next**: Notebook 04 — Epipolar Geometry & the Fundamental Matrix